In [18]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("arezaei81/heartcsv")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'heartcsv' dataset.
Path to dataset files: /kaggle/input/heartcsv


In [19]:
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split

import numpy as np
import pandas as pd

In [20]:
data = pd.read_csv('/root/.cache/kagglehub/datasets/arezaei81/heartcsv/versions/1/heart.csv')
data

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
298,57,0,0,140,241,0,1,123,1,0.2,1,0,3,0
299,45,1,3,110,264,0,1,132,0,1.2,1,0,3,0
300,68,1,0,144,193,1,1,141,0,3.4,1,2,3,0
301,57,1,0,130,131,0,1,115,1,1.2,1,1,3,0


In [21]:
from sklearn.preprocessing import StandardScaler

X = data[['age' , 'sex' , 'cp' , 'trestbps' , 'chol' , 'fbs' , 'restecg' , 'thalach' , 'exang' , 'oldpeak' , 'ca', 'thal']]
y = data['target']

# Scale numerical features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X = torch.from_numpy(X_scaled).float()
y = torch.from_numpy(y.values).float()

X_train , X_test , y_train , y_test = train_test_split(X,y , test_size=0.2,random_state=42)

Accuracy on the test set: 0.8033


In [22]:
class Model(nn.Module) :
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(12 , 15),
            nn.ReLU(),

            nn.Linear(15 , 45),
            nn.ReLU(),

            nn.Linear(45 , 100) ,
            nn.ReLU(),

            nn.Linear(100,50) ,
            nn.ReLU(),

            nn.Linear(50,1),
            nn.Sigmoid()
        )

    def forward(self , x):
        return self.network(x)

In [38]:
model_0 = Model()

loss_fn = nn.BCELoss()
opt = optim.Adam(model_0.parameters(),lr = 0.09)

In [39]:
for i in range(1000) :

    y_pred = model_0(X_train).squeeze()
    # Calculate loss
    loss = loss_fn(y_pred, y_train)

    # Zero gradients
    opt.zero_grad()

    # Backward pass
    loss.backward()

    # Optimize
    opt.step()

    if i % 1 == 0:
        print(f'Epoch {i}, Loss: {loss.item():.4f}')

Epoch 0, Loss: 0.6968
Epoch 1, Loss: 1.2227
Epoch 2, Loss: 2.2388
Epoch 3, Loss: 0.6766
Epoch 4, Loss: 0.6883
Epoch 5, Loss: 0.4945
Epoch 6, Loss: 0.4690
Epoch 7, Loss: 0.4039
Epoch 8, Loss: 0.3554
Epoch 9, Loss: 0.3563
Epoch 10, Loss: 0.3441
Epoch 11, Loss: 0.3401
Epoch 12, Loss: 0.3444
Epoch 13, Loss: 0.3321
Epoch 14, Loss: 0.3195
Epoch 15, Loss: 0.3109
Epoch 16, Loss: 0.3001
Epoch 17, Loss: 0.2884
Epoch 18, Loss: 0.2764
Epoch 19, Loss: 0.2688
Epoch 20, Loss: 0.2594
Epoch 21, Loss: 0.2491
Epoch 22, Loss: 0.2405
Epoch 23, Loss: 0.2277
Epoch 24, Loss: 0.2181
Epoch 25, Loss: 0.2070
Epoch 26, Loss: 0.1920
Epoch 27, Loss: 0.2406
Epoch 28, Loss: 0.2643
Epoch 29, Loss: 0.1926
Epoch 30, Loss: 0.2375
Epoch 31, Loss: 0.2000
Epoch 32, Loss: 0.1917
Epoch 33, Loss: 0.1791
Epoch 34, Loss: 0.1651
Epoch 35, Loss: 0.1660
Epoch 36, Loss: 0.1553
Epoch 37, Loss: 0.1466
Epoch 38, Loss: 0.1370
Epoch 39, Loss: 0.1307
Epoch 40, Loss: 0.1241
Epoch 41, Loss: 0.1168
Epoch 42, Loss: 0.1169
Epoch 43, Loss: 0.108

In [40]:
from sklearn.metrics import accuracy_score

# Set the model to evaluation mode
model_0.eval()

# Make predictions on the test set
with torch.no_grad():
    y_pred_test_prob = model_0(X_test).squeeze()
    y_pred_test_labels = (y_pred_test_prob > 0.5).float()

# Calculate accuracy
accuracy = accuracy_score(y_test.numpy(), y_pred_test_labels.numpy()) * 100
print(f'Accuracy on the test set: {accuracy:.4f}')

Accuracy on the test set: 81.9672
